# Copilot Cost Management (Graph beta)

Opt in only after granting application permission and attaching a Lakehouse. The beta API is not supported for production use.


In [ ]:
# Attach the destination Lakehouse and configure the app and Key Vault before running.
TENANT_ID = "00000000-0000-0000-0000-000000000000"
CLIENT_ID = "00000000-0000-0000-0000-000000000000"
KEY_VAULT_URL = "https://<your-vault>.vault.azure.net/"
CLIENT_SECRET_NAME = "consumption-central-azure-client-secret"
DAYS = 14  # Completed UTC days, from 1 to 366.


In [ ]:
import base64
import json
import math
import time
from datetime import date, datetime, timedelta, timezone
from urllib.error import HTTPError, URLError
from urllib.parse import urlencode, urlsplit
from urllib.request import HTTPRedirectHandler, Request, build_opener
from uuid import UUID
import requests
from pyspark.sql.types import DateType, DoubleType, StringType, StructField, StructType

class CollectionError(Exception):
    pass

GRAPH = "https://graph.microsoft.com"
ROOT = GRAPH + "/beta/copilot/costManagement"
SERVICE_FILE = "CopilotCostServiceDaily.csv"
FUNDING_FILE = "CopilotCostFundingMonthly.csv"
BALANCE_FILE = "CopilotCostBalance.csv"
SERVICE_HEADERS = ["Date", "ServiceId", "ConsumedQuantity", "Unit"]
FUNDING_HEADERS = ["Month", "FundingClass", "ConsumedQuantity", "Unit", "LatestSourceDateTime"]
BALANCE_HEADERS = ["AsOfUTC", "BillingMethod", "PurchasedQuantity", "ConsumedQuantity",
                   "RemainingQuantity", "Unit", "LastModifiedDateTime"]


def graph_url(url):
    parsed = urlsplit(url)
    if (parsed.scheme != "https" or parsed.netloc.lower() != "graph.microsoft.com"
            or not parsed.path.startswith("/beta/copilot/costManagement/")
            or parsed.fragment or "\\" in url or any(ord(c) <= 32 for c in url)):
        raise CollectionError("Invalid Graph request or continuation URL")
    return url


class NoRedirect(HTTPRedirectHandler):
    def redirect_request(self, req, fp, code, msg, headers, newurl):
        return None


class GraphClient:
    def __init__(self, token, opener=None, sleep=time.sleep):
        if not isinstance(token, str) or not token:
            raise CollectionError("Missing Graph access token")
        self.token, self.opener, self.sleep = token, opener or build_opener(NoRedirect()).open, sleep

    def get(self, url):
        url = graph_url(url)
        for attempt in range(4):
            try:
                request = Request(url, headers={"Authorization": "Bearer " + self.token,
                                                "Accept": "application/json"})
                with self.opener(request, timeout=90) as response:
                    data = json.load(response)
                if not isinstance(data, dict) or "error" in data:
                    raise CollectionError("Graph returned an invalid response")
                return data
            except HTTPError as exc:
                if exc.code not in (429, 500, 502, 503, 504) or attempt == 3:
                    raise CollectionError(f"Graph request failed with HTTP {exc.code}") from exc
                retry = exc.headers.get("Retry-After")
                if retry is not None and (not retry.isdecimal() or int(retry) > 3600):
                    raise CollectionError("Graph returned an invalid Retry-After") from exc
                self.sleep(int(retry) if retry is not None else 2 ** (attempt + 1))
            except (URLError, OSError, ValueError, UnicodeDecodeError) as exc:
                raise CollectionError(f"Graph response could not be read: {type(exc).__name__}") from exc
        raise CollectionError("Graph retry limit exceeded")

    def list(self, url):
        rows, seen = [], set()
        while url:
            graph_url(url)
            if url in seen or len(seen) >= 10000:
                raise CollectionError("Repeated or excessive Graph continuation pages")
            seen.add(url)
            page = self.get(url)
            if not isinstance(page.get("value"), list):
                raise CollectionError("Graph collection has no value array")
            rows.extend(page["value"])
            url = page.get("@odata.nextLink")
            if url is not None and not isinstance(url, str):
                raise CollectionError("Invalid Graph continuation")
        return rows


def quantity(record, name, optional=False):
    value = record.get(name)
    if value is None and optional:
        return ""
    if isinstance(value, bool) or not isinstance(value, (int, float)) or not math.isfinite(value):
        raise CollectionError(f"Invalid {name} in Graph response")
    return value


def text(record, name):
    value = record.get(name)
    if not isinstance(value, str) or not value:
        raise CollectionError(f"Invalid {name} in Graph response")
    return value


def unit(record):
    value = text(record, "unit")
    if value != "copilotCredit":
        raise CollectionError("Unexpected credit unit in Graph response")
    return value


def optional_status(record, field):
    status = record.get("dataStatus")
    if status is None:
        return ""
    if not isinstance(status, dict):
        raise CollectionError("Invalid dataStatus in Graph response")
    value = status.get(field)
    if value is not None and not isinstance(value, str):
        raise CollectionError("Invalid freshness timestamp in Graph response")
    return value or ""


def distinct(rows, key):
    result = {}
    for row in rows:
        identity = tuple(row[name] for name in key)
        if identity in result and result[identity] != row:
            raise CollectionError("Conflicting Graph records in the same reporting window")
        result[identity] = row
    return list(result.values())


def collect(client, start, end, as_of):
    if start >= end:
        raise CollectionError("The collection window must contain at least one completed UTC day")
    services = []
    day = start
    while day < end:
        url = ROOT + "/usageReports/serviceUsageRecords?" + urlencode(
            {"startDate": day.isoformat(), "endDate": (day + timedelta(days=1)).isoformat()})
        for item in client.list(url):
            if not isinstance(item, dict):
                raise CollectionError("Invalid Graph service record")
            services.append({"Date": day.isoformat(), "ServiceId": text(item, "serviceId"),
                             "ConsumedQuantity": quantity(item, "consumedQuantity"), "Unit": unit(item)})
        day += timedelta(days=1)

    funding = []
    month = start.replace(day=1)
    while month < end:
        url = ROOT + "/usageReports/fundingUsageRecords?" + urlencode({"month": month.strftime("%Y-%m")})
        for item in client.list(url):
            if not isinstance(item, dict):
                raise CollectionError("Invalid Graph funding record")
            funding.append({"Month": month.isoformat(), "FundingClass": text(item, "fundingClass"),
                            "ConsumedQuantity": quantity(item, "consumedQuantity"), "Unit": unit(item),
                            "LatestSourceDateTime": optional_status(item, "latestSourceDateTime")})
        month = (month.replace(day=28) + timedelta(days=4)).replace(day=1)

    snapshot = client.get(ROOT + "/getTenantCreditBalance")
    balances = snapshot.get("billingMethodBalances")
    if not isinstance(balances, list):
        raise CollectionError("Graph balance response has no billingMethodBalances array")
    balance = []
    for item in balances:
        if not isinstance(item, dict):
            raise CollectionError("Invalid Graph balance record")
        balance.append({"AsOfUTC": as_of, "BillingMethod": text(item, "billingMethod"),
                        "PurchasedQuantity": quantity(item, "purchasedQuantity", optional=True),
                        "ConsumedQuantity": quantity(item, "consumedQuantity"),
                        "RemainingQuantity": quantity(item, "remainingQuantity", optional=True),
                        "Unit": unit(item),
                        "LastModifiedDateTime": optional_status(item, "lastModifiedDateTime")})
    return {
        SERVICE_FILE: (SERVICE_HEADERS, distinct(services, ["Date", "ServiceId"])),
        FUNDING_FILE: (FUNDING_HEADERS, distinct(funding, ["Month", "FundingClass"])),
        BALANCE_FILE: (BALANCE_HEADERS, distinct(balance, ["BillingMethod"])),
    }


def app_token(payload):
    token = payload.get("accessToken") if isinstance(payload, dict) else None
    try:
        encoded = token.split(".")[1]
        claims = json.loads(base64.urlsafe_b64decode(encoded + "=" * (-len(encoded) % 4)))
    except (AttributeError, IndexError, ValueError, UnicodeDecodeError, TypeError) as exc:
        raise CollectionError("Azure CLI did not return a readable Graph access token") from exc
    roles = claims.get("roles") if isinstance(claims, dict) else None
    if (not isinstance(claims, dict) or claims.get("scp") or not isinstance(roles, list)
            or "CopilotCostManagement.Read.All" not in roles):
        raise CollectionError("The Azure CLI login must be an app with the consented "
                              "CopilotCostManagement.Read.All application permission")
    return token


In [ ]:
for identifier in (TENANT_ID, CLIENT_ID):
    if UUID(identifier).int == 0:
        raise ValueError("Configure the tenant and application client IDs before running")
if not isinstance(DAYS, int) or isinstance(DAYS, bool) or not 1 <= DAYS <= 366:
    raise ValueError("DAYS must be between 1 and 366")
secret = notebookutils.credentials.getSecret(KEY_VAULT_URL, CLIENT_SECRET_NAME)
response = requests.post(
    f"https://login.microsoftonline.com/{TENANT_ID}/oauth2/v2.0/token",
    data={"grant_type": "client_credentials", "client_id": CLIENT_ID,
          "client_secret": secret, "scope": GRAPH + "/.default"},
    timeout=90, allow_redirects=False)
if response.status_code != 200:
    raise CollectionError(f"Graph token request failed with HTTP {response.status_code}")
try:
    token = response.json().get("access_token")
except ValueError as exc:
    raise CollectionError("Graph token response was not JSON") from exc
token = app_token({"accessToken": token})
del secret, response

now = datetime.now(timezone.utc)
end = now.date()
start = end - timedelta(days=DAYS)
datasets = collect(GraphClient(token), start, end, now.isoformat())
del token

def schema(fields):
    return StructType([StructField(name, kind, True) for name, kind in fields])

service_schema = schema([("Date", DateType()), ("ServiceId", StringType()),
                         ("ConsumedQuantity", DoubleType()), ("Unit", StringType())])
funding_schema = schema([("Month", DateType()), ("FundingClass", StringType()),
                         ("ConsumedQuantity", DoubleType()), ("Unit", StringType()),
                         ("LatestSourceDateTime", StringType())])
balance_schema = schema([("AsOfUTC", StringType()), ("BillingMethod", StringType()),
                         ("PurchasedQuantity", DoubleType()), ("ConsumedQuantity", DoubleType()),
                         ("RemainingQuantity", DoubleType()), ("Unit", StringType()),
                         ("LastModifiedDateTime", StringType())])

def number(value):
    return None if value == "" else float(value)

services = [(date.fromisoformat(r["Date"]), r["ServiceId"], number(r["ConsumedQuantity"]), r["Unit"])
            for r in datasets[SERVICE_FILE][1]]
funding = [(date.fromisoformat(r["Month"]), r["FundingClass"], number(r["ConsumedQuantity"]),
            r["Unit"], r["LatestSourceDateTime"]) for r in datasets[FUNDING_FILE][1]]
balances = [(r["AsOfUTC"], r["BillingMethod"], number(r["PurchasedQuantity"]),
             number(r["ConsumedQuantity"]), number(r["RemainingQuantity"]), r["Unit"],
             r["LastModifiedDateTime"]) for r in datasets[BALANCE_FILE][1]]
writes = [
    ("copilot_cost_service_daily", spark.createDataFrame(services, service_schema), "Date", start),
    ("copilot_cost_funding_monthly", spark.createDataFrame(funding, funding_schema), "Month", start.replace(day=1)),
    ("copilot_cost_balance", spark.createDataFrame(balances, balance_schema), None, None),
]

# Check every existing table before the first write. Each table write is atomic, but the
# three tables are not one transaction. If a write fails, stop refresh and rerun the notebook.
for table, frame, column, cut in writes:
    if spark.catalog.tableExists(table):
        stored = {name.lower(): kind for name, kind in spark.table(table).dtypes}
        for name, kind in frame.dtypes:
            if name.lower() in stored and stored[name.lower()] != kind:
                raise CollectionError(f"Incompatible type for {table}.{name}; nothing was written")

for table, frame, column, cut in writes:
    exists = spark.catalog.tableExists(table)
    if not exists:
        frame.write.format("delta").saveAsTable(table)
    elif column:
        predicate = f"`{column}` >= DATE '{cut.isoformat()}' OR `{column}` IS NULL"
        (frame.write.format("delta").mode("overwrite").option("replaceWhere", predicate)
         .option("mergeSchema", "true").saveAsTable(table))
    else:
        frame.write.format("delta").mode("overwrite").option("mergeSchema", "true").saveAsTable(table)
    print(f"Updated {table}: {frame.count():,} rows")
print("Service usage, funding usage and balance are separate credit views; do not add them together.")
